# 02 — Generating test sets

Building a custom prompt evaluation workflow starts with creating a solid prompt and then generating test data to see how well it performs. Let's walk through setting up an evaluation system for a prompt that helps users write AWS-specific code.

An evaluation dataset contains inputs that we'll feed into our prompt. For each combination of prompt and input, we'll run the prompt and analyze the results.

Our dataset will be an array of JSON objects, where each object contains a "task" property describing what we want Claude to accomplish. We can either create this dataset by hand or generate it automatically using Claude.

Since we're generating test data, this is a perfect opportunity to use a faster model like Haiku instead of the full Claude model.

In [3]:
# Add repo root to path
import sys
sys.path.append("../..")

In [4]:
# Get the client and model from utils
from src.utils import get_client, get_model

client = get_client()
model = get_model(default_model="claude-haiku-4-5", use_default=True)

In [5]:
# Helper functions
from src.utils import chat, add_user_message, add_assistant_message

## 1. Generating Test Data with Code

In [6]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(
        messages=messages,
        client=client,
        model=model,
        temperature=0.0,
        stop_sequences=["```"]
    )
    return json.loads(text)

In [7]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL in the format 's3://bucket-name.region.amazonaws.com'. The function should return the region name."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'. Include the necessary Action, Resource, and Effect fields."}, {'task': "Write a regular expression that matches valid AWS EC2 instance IDs, which follow the pattern of 'i-' followed by exactly 17 hexadecimal characters (e.g., i-0a1b2c3d4e5f6g7h8)."}]


## 2. Saving the Dataset

In [8]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)